# WDL — Module 3 · Session 11 — Totals, averages and the normal
### Reading notebook

| | |
|---|---|
| **Course** | Water Data Literacy: Rainfall, Floods and Climate Change |
| **Edition** | Udemy |
| **Instructor** | Dr Arpan Pradhan |
| **Module** | Module 3 — Reading Rainfall |
| **Session** | Session 11 of 30 |
| **Topics** | Annual and seasonal totals; long-period average (LPA); mean vs median |
| **Water hook** | *What does a "normal monsoon" mean?* |

---
> **How to use this notebook.** No coding needed. Read each explanation, then press ▶ on the box below it
> (or click the box and press `Shift + Enter`). Some boxes have sliders, number boxes or dropdowns: change them, then press ▶ again.
> Go to **File → Save a copy in Drive** first, so the notebook is yours.

## Why this session matters
"Normal monsoon", "below normal", "LPA": every season these words decide headlines and farm plans. This session shows
how a **normal** is calculated, why the **average** can mislead, and how much of the year's water comes in the monsoon.

In [ ]:
#@title ▶ Step 0 · Get ready (press ▶ once before anything else)
import os, json, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
_orig_bar = plt.Axes.bar
def _bar_int_years(self, x, *a, **k):        # whole-number year ticks on bar charts of years
    out = _orig_bar(self, x, *a, **k)
    try:
        xs = np.asarray(x)
        if xs.dtype.kind in "iu" and xs.min() > 1800: self.xaxis.set_major_locator(MaxNLocator(integer=True))
    except Exception: pass
    return out
plt.Axes.bar = _bar_int_years
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110, "font.size": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, GREY, TEAL, GOLD = "#2C6E9B", "#DD8452", "#C44E52", "#8C8C8C", "#1F7A8C", "#C9962E"
AUTO = os.environ.get("WDL_SELFTEST") == "1"

def ok(msg="Correct!"):  display(Markdown(f"✅ **{msg}**"))
def no(msg="Not quite. Check your working and try again."): display(Markdown(f"❌ {msg}"))
def _num(x):
    try: return float(str(x).replace(",", "").replace("%", "").strip())
    except (TypeError, ValueError): return None
def close(a, b, tol=0.02):
    return a is not None and abs(a - b) <= max(abs(b) * tol, 0.01)
def check_num(ans, exp, tol=0.02, fmt="{:,.1f}", hint=""):
    if AUTO: ans = exp
    v = _num(ans)
    if v is None: no("Type your answer in the box first (numbers only)."); return False
    if close(v, float(exp), tol): ok("Correct: " + fmt.format(exp)); return True
    no("Not quite. " + hint); return False
def check_choice(ans, exp, explain="", hint=""):
    if AUTO: ans = exp
    if ans in ("", "choose"): no("Choose an answer from the dropdown first."); return False
    if ans == exp: ok(("Correct. " + explain).strip()); return True
    no(("Not quite. " + hint).strip()); return False

def make_sample_district():
    """Illustrative, synthetic daily rainfall for a South Indian district, 1981-2024."""
    rng = np.random.default_rng(1981)
    days = pd.date_range("1981-01-01", "2024-12-31", freq="D")
    p_wet = np.array([0.03, 0.03, 0.06, 0.15, 0.30, 0.35, 0.42, 0.42, 0.45, 0.45, 0.28, 0.08])
    mu = np.array([4, 4, 7, 11, 13, 8, 8, 9, 13, 14, 12, 6], float)
    years = np.arange(1981, 2025)
    yf = dict(zip(years, np.exp(rng.normal(0, 0.17, len(years)))))
    m = days.month.values - 1
    t = days.year.values - 1981
    yfac = np.array([yf[y] for y in days.year.values])
    p = np.clip(p_wet[m] * (1 - 0.0025 * t) * yfac ** 0.5, 0, 0.95)
    inten = mu[m] * (1 + 0.005 * t) * yfac ** 0.5
    wet = rng.random(len(days)) < p
    amt = rng.gamma(0.7, inten / 0.7)
    rain = np.round(np.where(wet, amt, 0.0), 1)
    for d, v in [("1991-11-14", 142.0), ("2005-10-24", 186.2), ("2017-08-14", 151.4), ("2022-09-05", 168.9)]:
        rain[days.get_loc(pd.Timestamp(d))] = v
    return pd.Series(rain, index=days, name="rain_mm")

def make_messy_record(sample):
    """Three years of the sample record with typical gauge-record problems added."""
    df = sample["2018":"2020"].rename("rain_mm").reset_index()
    df.columns = ["date", "rain_mm"]
    flag_dates = ["2018-02-03", "2018-05-19", "2018-08-07", "2018-11-25", "2019-01-14",
                  "2019-04-02", "2019-09-18", "2019-12-30", "2020-06-06", "2020-10-11"]
    blank_dates = ["2018-07-21", "2019-03-09", "2020-02-17", "2020-08-28"]
    df.loc[df.date.isin(pd.to_datetime(flag_dates)), "rain_mm"] = -999.0
    df.loc[df.date.isin(pd.to_datetime(blank_dates)), "rain_mm"] = np.nan
    df.loc[df.date == "2018-09-12", "rain_mm"] = 999.9      # typing error
    df.loc[df.date == "2020-04-23", "rain_mm"] = -3.0       # impossible negative
    df = df[~df.date.between("2019-07-01", "2019-07-12")]  # 12 days never recorded
    dup = df[df.date.isin(pd.to_datetime(["2018-06-15", "2019-10-03", "2020-11-20"]))]
    df = pd.concat([df, dup]).sort_values("date", kind="stable").reset_index(drop=True)
    df["date"] = df.date.dt.strftime("%Y-%m-%d")
    return df

SAMPLE_URL = "https://raw.githubusercontent.com/PradhanArpan/waterliteracy/main/data/sample_district_daily_1981_2024.csv"
try:
    sample = pd.read_csv(SAMPLE_URL, parse_dates=["date"], index_col="date")["rain_mm"]
except Exception:
    sample = make_sample_district()   # works offline too: same numbers

def find_place(name):
    """Place name -> (lat, lon, label) using Open-Meteo geocoding. Prefers places in India."""
    parts = [p.strip() for p in str(name).split(",") if p.strip()]
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": parts[0], "count": 10, "language": "en", "format": "json"}, timeout=20)
    res = r.json().get("results", []) or []
    res = [x for x in res if x.get("country_code") == "IN"] or res
    if len(parts) > 1:
        res = [x for x in res if parts[1].lower() in str(x.get("admin1", "")).lower()] or res
    if not res: raise ValueError(f"No place found for '{name}'")
    x = res[0]
    return x["latitude"], x["longitude"], f'{x["name"]}, {x.get("admin1", "")}'.strip(", ")

def fetch_power(lat, lon, start=1981, end=2024):
    """Daily rainfall (mm/day) from NASA POWER (PRECTOTCORR) for one point."""
    r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", timeout=90, params={
        "parameters": "PRECTOTCORR", "community": "AG", "latitude": round(lat, 4), "longitude": round(lon, 4),
        "start": f"{start}0101", "end": f"{end}1231", "format": "JSON"})
    r.raise_for_status()
    d = r.json()["properties"]["parameter"]["PRECTOTCORR"]
    s = pd.Series(d, dtype=float)
    s.index = pd.to_datetime(s.index, format="%Y%m%d")
    s[s < -98] = np.nan
    return s.round(1).rename("rain_mm")

def fetch_era5(lat, lon, start="2021-01-01", end="2021-12-31"):
    """Daily rainfall (mm/day) from the ERA5 reanalysis via Open-Meteo."""
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params={
        "latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
        "daily": "precipitation_sum", "timezone": "Asia/Kolkata"})
    r.raise_for_status()
    d = r.json()["daily"]
    return pd.Series(d["precipitation_sum"], index=pd.to_datetime(d["time"]), dtype=float).rename("rain_mm")

def load_rain(source, place="", lat=0.0, lon=0.0):
    """Returns (daily rainfall Series, label, lat, lon). Falls back to the sample district."""
    if source.startswith("Sample") or AUTO:
        return sample, "Sample district (synthetic)", None, None
    try:
        if not (lat and lon):
            lat, lon, place = find_place(place)
        cache = f"rain_{lat:.3f}_{lon:.3f}.csv"
        if os.path.exists(cache):
            s = pd.read_csv(cache, parse_dates=["date"], index_col="date")["rain_mm"]
        else:
            s = fetch_power(lat, lon)
            s.rename_axis("date").to_csv(cache)
        return s, f"{place} ({lat:.2f}°N, {lon:.2f}°E) · NASA POWER", lat, lon
    except Exception as e:
        display(Markdown(f"⚠️ Could not download data for **{place or 'your place'}** ({type(e).__name__}). "
                         "Using the sample district instead. Check the place name or type latitude and longitude."))
        return sample, "Sample district (synthetic)", None, None

def annual(s):  return s.groupby(s.index.year).sum(min_count=300)
def monsoon(s):
    m = s[s.index.month.isin([6, 7, 8, 9])]
    return m.groupby(m.index.year).sum(min_count=100)
display(Markdown("✅ **Ready.** Tools loaded. Now run **Step 1** to choose your data."))

---
## Step 1 · Choose your data
> **About the data.** *Sample district* is an illustrative, synthetic 44-year record (1981–2024) made for practice;
> it is not an official record, and answers in the practice tasks and the Sheet use it.
> *My place* downloads real daily rainfall for any location from **NASA POWER** (satellite and model based, from 1981).
> Type a town or district name, or enter latitude and longitude.

In [ ]:
#@title ▶ Step 1 · Choose your data { run: "auto" }
data_source = "Sample district (works offline)" #@param ["Sample district (works offline)", "My place (live download from NASA POWER)"]
place = "Mysuru, Karnataka" #@param {type:"string"}
latitude = 0 #@param {type:"number"}
longitude = 0 #@param {type:"number"}
rain, label, LAT, LON = load_rain(data_source, place, latitude, longitude)
display(Markdown(f"**Data:** {label} · {rain.index.min():%d %b %Y} to {rain.index.max():%d %b %Y} · "
                 f"{len(rain):,} days"))

In [ ]:
#@title ▶ Step 2 · Prepare the tables (press ▶)
A = annual(rain)                                    # annual totals
SW = monsoon(rain)                                  # June–September totals
NORMAL = A.loc[1991:2020].mean()                    # 30-year normal, WMO standard period
NORMAL_SW = SW.loc[1991:2020].mean()
def imd_category(dep):
    if dep <= -99.5: return "No rain"
    if dep <= -60: return "Large deficient"
    if dep <= -20: return "Deficient"
    if dep < 20: return "Normal"
    if dep < 60: return "Excess"
    return "Large excess"
CAT_COLOURS = {"Large deficient": "#8E2C2C", "Deficient": RED, "Normal": GREY, "Excess": BLUE, "Large excess": "#1B3F66", "No rain": "black"}
INTENSITY = [("Very light", 0.1, 2.4), ("Light", 2.5, 15.5), ("Moderate", 15.6, 64.4), ("Heavy", 64.5, 115.5),
             ("Very heavy", 115.6, 204.4), ("Extremely heavy", 204.5, 10000)]
def intensity_class(x):
    for name, lo, hi in INTENSITY:
        if lo <= x <= hi: return name
    return "No rain"
AMAX = rain.groupby(rain.index.year).max()          # annual maximum daily rainfall
display(Markdown(f"Tables ready. 1991–2020 normal: **{NORMAL:,.0f} mm** per year, **{NORMAL_SW:,.0f} mm** in the June–September monsoon."))

---
## Part 1 · What is a "normal"?
A **normal** is the average over a long, fixed period. The World Meteorological Organization's standard is **30 years**;
the current standard period is **1991–2020**. IMD calls its national normal the **Long Period Average (LPA)**.
Press ▶ to see each year against the normal.

In [ ]:
#@title ▶ Annual totals and the normal { run: "auto" }
normal_start = 1991 #@param {type:"slider", min:1981, max:1995, step:1}
normal_end = 2020 #@param {type:"slider", min:2000, max:2024, step:1}
nm = A.loc[normal_start:normal_end].mean()
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(A.index, A.values, color=[BLUE if v >= nm else ORANGE for v in A.values], width=0.8)
ax.axhline(nm, color="black", lw=1.4); ax.text(A.index[0], nm * 1.03, f"Normal {normal_start}–{normal_end}: {nm:,.0f} mm", fontsize=10)
ax.axvspan(normal_start - 0.5, normal_end + 0.5, color=GREY, alpha=0.08)
ax.set_ylabel("Rainfall (mm per year)"); ax.set_title(f"{label.split(' · ')[0]}: years above (blue) and below (orange) the normal")
plt.tight_layout(); plt.show()

Move the sliders: the normal changes a little with the period chosen. That is why reports always say *which* normal they use.

---
## Part 2 · Mean or median?
The **mean** adds everything and divides. The **median** is the middle year when all years are sorted. A few very wet years
pull the mean up, so in most places the median year is a little drier than the mean. Press ▶.

In [ ]:
#@title ▶ Mean vs median
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.hist(A.values, bins=12, color=BLUE, alpha=0.75, edgecolor="white")
ax.axvline(A.mean(), color=ORANGE, lw=2); ax.axvline(A.median(), color=TEAL, lw=2, ls="--")
ax.text(A.mean(), ax.get_ylim()[1] * 0.92, f"  mean {A.mean():,.0f}", color=ORANGE)
ax.text(A.median(), ax.get_ylim()[1] * 0.80, f"  median {A.median():,.0f}", color=TEAL)
ax.set_xlabel("Annual rainfall (mm)"); ax.set_ylabel("Number of years"); ax.set_title("How annual totals are spread")
plt.tight_layout(); plt.show()
display(Markdown(f"Wettest year **{A.idxmax()}** ({A.max():,.0f} mm) · driest year **{A.idxmin()}** ({A.min():,.0f} mm) · "
                 f"the wettest year had **{A.max()/A.min():.1f} times** the rain of the driest."))

---
## Part 3 · Where the year's water comes from
Press ▶ to see the average for each month and each IMD season.

In [ ]:
#@title ▶ The average year { run: "auto" }
show = "By month" #@param ["By month", "By season"]
mon = rain.groupby([rain.index.year, rain.index.month]).sum().groupby(level=1).mean()
if show == "By month":
    ax = pd.Series(mon.values, index=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]).plot(kind="bar", color=BLUE, rot=0)
    ax.bar_label(ax.containers[0], fmt="%.0f", fontsize=9); ax.set_title("Average rainfall by month (mm)")
else:
    sea = pd.Series({"Winter\n(Jan–Feb)": mon[[1, 2]].sum(), "Pre-monsoon\n(Mar–May)": mon[[3, 4, 5]].sum(),
                     "SW monsoon\n(Jun–Sep)": mon[[6, 7, 8, 9]].sum(), "NE monsoon\n(Oct–Dec)": mon[[10, 11, 12]].sum()})
    ax = sea.plot(kind="bar", color=[GREY, ORANGE, BLUE, TEAL], rot=0)
    ax.bar_label(ax.containers[0], labels=[f"{v:,.0f} mm\n({v/sea.sum()*100:.0f}%)" for v in sea], fontsize=9)
    ax.set_title("Average rainfall by IMD season"); ax.set_ylim(0, sea.max() * 1.25)
ax.set_ylabel("mm"); plt.tight_layout(); plt.show()

For most of India, the **southwest monsoon (June–September)** brings 70–90% of the year's rain. In parts of Tamil Nadu,
coastal Andhra Pradesh and south interior Karnataka, the **northeast monsoon (October–December)** matters a great deal too.
Switch Step 1 to your own place and compare.

### ⏭️ Next
Open the **Practice activity** and compute the normal yourself.